# Audio generation with AudioLDM 2

Generate sound effects, music and speech-like audio from text using the Diffusers AudioLDM 2 pipeline. Install `diffusers`, `transformers`, `accelerate`, `soundfile`, and PyTorch; model weights download on first use and generation can take a long time. Check each model card for its license before use.

The standard pipeline supports text-conditioned generation. It does not accept the `audio`, `mask_time_indices`, or `strength` arguments used by the previous inpainting and audio-variation examples, so those non-working examples have been removed. See the [official AudioLDM 2 API](https://huggingface.co/docs/diffusers/en/api/pipelines/audioldm2). The remaining examples have not been rerun here.

**Data:** No external dataset. Prompts are defined in the notebook and the model generates the audio.

In [1]:
import torch
import numpy as np
import soundfile as sf
import librosa
import IPython.display as ipd
import os

from diffusers import AudioLDM2Pipeline

def save_and_play(audio: np.ndarray, sample_rate: int, filename: str):
    os.makedirs("outputs", exist_ok=True)
    path = f"outputs/{filename}"
    sf.write(path, audio, sample_rate)
    print(f"Сохранено: {path}")
    return ipd.display(ipd.Audio(audio, rate=sample_rate))

def play_file(filename: str):
    path = f"outputs/{filename}" if not filename.startswith("outputs/") else filename
    audio, sr = sf.read(path)
    print(f"{path} | sr={sr} | duration={len(audio)/sr:.2f}s")
    return ipd.display(ipd.Audio(audio, rate=sr))

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
TARGET_SR = 16000
print(f"Устройство: {DEVICE} | dtype: {DTYPE}")

## Text-to-audio

Generate sound effects or ambience with `cvssp/audioldm2`.

In [2]:
# Authenticate outside the notebook, for example with `hf auth login`, if the model requires it.

In [4]:
pipe_audio = AudioLDM2Pipeline.from_pretrained(
    "cvssp/audioldm2",
    torch_dtype=DTYPE,
).to(DEVICE)

pipe_audio.enable_attention_slicing()

Fetching 26 files:   8%|▊         | 2/26 [15:11<3:02:22, 455.94s/it]


In [ ]:
# Параметры
prompt          = "Thunderstorm with heavy rain and distant thunder"
negative_prompt = "music, speech, low quality"
audio_length    = 10.0
num_steps       = 200    # больше шагов = лучше качество, медленнее
guidance_scale  = 3.5    # CFG (3–4 оптимально для audio)
num_waveforms   = 2

# Генерация
result = pipe_audio(
    prompt,
    negative_prompt          = negative_prompt,
    audio_length_in_s        = audio_length,
    num_inference_steps      = num_steps,
    guidance_scale           = guidance_scale,
    num_waveforms_per_prompt = num_waveforms,
)

for i, audio in enumerate(result.audios):
    print(f"\n--- Вариант {i+1} ---")
    save_and_play(audio, sample_rate=TARGET_SR, filename=f"t2a_variant_{i+1}.wav")

## Text-to-music

The `cvssp/audioldm2-music` checkpoint is trained for music generation.

In [ ]:
if "pipe_audio" in globals():
    del pipe_audio

if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

pipe_music = AudioLDM2Pipeline.from_pretrained(
    "cvssp/audioldm2-music",
    torch_dtype=DTYPE,
).to(DEVICE)

pipe_music.enable_attention_slicing()

Fetching 26 files:   8%|▊         | 2/26 [03:25<41:02, 102.60s/it]


In [ ]:
music_prompt = "Lo-fi hip hop, chill beats, piano melody, vinyl crackle, 80 BPM"
neg_music    = "noise, distortion, low quality, speech"

result_music = pipe_music(
    music_prompt,
    negative_prompt          = neg_music,
    audio_length_in_s        = 10.0,
    num_inference_steps      = 200,
    guidance_scale           = 3.5,
    num_waveforms_per_prompt = 2,
)

for i, audio in enumerate(result_music.audios):
    print(f"\n--- Вариант {i+1} ---")
    save_and_play(audio, sample_rate=TARGET_SR, filename=f"music_variant_{i+1}.wav")

## Speech-like audio

The following prompt requests speech-like sound; it does not specify a transcript or guarantee intelligible words.

In [ ]:
if "pipe_music" in globals():
    del pipe_music

if DEVICE == "cuda":
    torch.cuda.empty_cache()
elif DEVICE == "mps":
    torch.mps.empty_cache()

pipe_large = AudioLDM2Pipeline.from_pretrained(
    "cvssp/audioldm2-large",
    torch_dtype=DTYPE,
).to(DEVICE)

pipe_large.enable_attention_slicing()

In [ ]:
speech_prompt   = "A male speaker with a deep voice reads a news report clearly and professionally"
neg_speech      = "music, noise, echo, low quality"

result_speech = pipe_large(
    speech_prompt,
    negative_prompt          = neg_speech,
    audio_length_in_s        = 5.0,
    num_inference_steps      = 200,
    guidance_scale           = 2.0,   # Низкий guidance лучше для речи
    num_waveforms_per_prompt = 2,
)

for i, audio in enumerate(result_speech.audios):
    print(f"\n--- Вариант {i+1} ---")
    save_and_play(audio, sample_rate=TARGET_SR, filename=f"speech_variant_{i+1}.wav")

## Multiple prompts

Generate a separate audio example for each prompt.

In [ ]:
prompts = [
    "Dog barking in a park",
    "Coffee machine brewing espresso",
    "Crowd cheering at a stadium",
]

for idx, prompt in enumerate(prompts):
    result = pipe_large(
        prompt,
        negative_prompt          = "music, low quality",
        audio_length_in_s        = 5.0,
        num_inference_steps      = 100,   # быстро для демо
        guidance_scale           = 3.5,
        num_waveforms_per_prompt = 1,
    )
    print(f"\n[{idx+1}] {prompt}")
    save_and_play(result.audios[0], sample_rate=TARGET_SR, filename=f"batch_{idx+1}.wav")

## Parameters and checkpoints

- `num_inference_steps` controls the denoising steps and affects runtime.
- `guidance_scale` controls the strength of text conditioning.
- `audio_length_in_s` sets the requested duration.
- `negative_prompt` supplies an optional description of unwanted sounds.
- `num_waveforms_per_prompt` sets how many outputs to generate.

See the [pipeline documentation](https://huggingface.co/docs/diffusers/en/api/pipelines/audioldm2) and the model cards for current requirements and licenses.

In [ ]:
# Список всех сгенерированных файлов
if os.path.exists("outputs"):
    for f in sorted(os.listdir("outputs")):
        print(f"outputs/{f}")